# MIC90 Prediction Using Multiple Machine Learning Models

## Dataset
input/MIC90_normalised_data.xlsx

## Preprocessing Sheets
1. Min-Max Scaling
2. MaxAbs Scaling
3. Log Transformation
4. Box-Cox Transformation

## Models Used
1. Multiple-linear regression
2. LASSO regression
3. Decision tree and random forest
4. Support vector machines
5. Gradient boosting

## Evaluation Metrics
- R²
- Adjusted R²
- RMSE
- MAE
- CV_R²

In [1]:
# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

import os
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    cross_val_score
)

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import (
    LinearRegression,
    Lasso
)

from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

from sklearn.svm import SVR

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

from openpyxl.styles import (
    Font,
    PatternFill,
    Border,
    Side,
    Alignment
)

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [2]:
# ============================================================
# LOAD DATASET
# ============================================================

file_path = "input/MIC90_normalised_data.xlsx"

# Read first sheet
df = pd.read_excel(
    file_path,
    sheet_name=0
)

df.columns = [
    "MIC90",
    "LogP",
    "MR",
    "HL",
    "tPSA",
    "CLogP",
    "CMR",
    "LogS"
]

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (181, 8)


,MIC90,LogP,MR,HL,tPSA,CLogP,CMR,LogS
0,0.000306,0.500562,0.576498,0.654308,0.200892,0.394928,0.384296,0.618881
1,0.006326,0.511811,0.608758,0.647764,0.200892,0.449385,0.426113,0.593240
2,0.008265,0.364454,0.542087,0.735005,0.232107,0.208517,0.333679,0.708625
3,0.015612,0.382452,0.544537,0.731189,0.232107,0.271714,0.335076,0.681818
4,0.007755,0.382452,0.544537,0.731189,0.232107,0.271714,0.335076,0.681818


In [3]:
# ============================================================
# DEFINE FEATURES AND TARGET
# ============================================================

X = df[
    [
        "LogP",
        "MR",
        "HL",
        "tPSA",
        "CLogP",
        "CMR",
        "LogS"
    ]
]

y = df["MIC90"]

print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Matrix Shape: (181, 7)
Target Shape: (181,)


In [4]:
# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Samples:", len(X_train))
print("Testing Samples :", len(X_test))

Training Samples: 144
Testing Samples : 37


In [5]:
# ============================================================
# FEATURE SCALING
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)

print("Scaling Completed")

Scaling Completed


In [6]:
# ============================================================
# DEFINE MODELS
# ============================================================

models = {

    "Multiple linear regression":
        LinearRegression(),

    "LASSO regression":
        Lasso(
            alpha=0.01
        ),

    "Decision tree":
        DecisionTreeRegressor(
            max_depth=5,
            random_state=42
        ),

    "Random forest":
        RandomForestRegressor(
            n_estimators=300,
            random_state=42
        ),

    "Support vector machines":
        SVR(
            kernel="rbf",
            C=10
        ),

    "Gradient boosting":
        GradientBoostingRegressor(
            n_estimators=200,
            random_state=42
        )
}

print("Total Models =", len(models))

Total Models = 6


In [7]:
# ============================================================
# MODEL TRAINING AND EVALUATION
# ============================================================

results = []

predictions_df = pd.DataFrame()

predictions_df["Actual_MIC90"] = y_test.values

for name, model in models.items():

    print("\n" + "="*60)
    print(name)
    print("="*60)

    if name in [
        "Multiple linear regression",
        "LASSO regression",
        "Support vector machines"
    ]:

        model.fit(
            X_train_scaled,
            y_train
        )

        y_pred = model.predict(
            X_test_scaled
        )

        cv_r2 = cross_val_score(
            model,
            X_train_scaled,
            y_train,
            cv=5,
            scoring="r2"
        ).mean()

    else:

        model.fit(
            X_train,
            y_train
        )

        y_pred = model.predict(
            X_test
        )

        cv_r2 = cross_val_score(
            model,
            X_train,
            y_train,
            cv=5,
            scoring="r2"
        ).mean()

    # Metrics

    r2 = r2_score(
        y_test,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            y_pred
        )
    )

    mae = mean_absolute_error(
        y_test,
        y_pred
    )

    # Adjusted R²

    n = len(y_test)

    p = X.shape[1]

    adj_r2 = 1 - (
        (1-r2)*(n-1)/(n-p-1)
    )

    results.append([
        name,
        r2,
        adj_r2,
        rmse,
        mae,
        cv_r2
    ])

    predictions_df[name] = y_pred

    print("R²      :", round(r2,4))
    print("Adj R²  :", round(adj_r2,4))
    print("RMSE    :", round(rmse,4))
    print("MAE     :", round(mae,4))
    print("CV_R²   :", round(cv_r2,4))


Multiple linear regression
R²      : 0.3934
Adj R²  : 0.2469
RMSE    : 0.1717
MAE     : 0.1059
CV_R²   : -670.988

LASSO regression
R²      : 0.3248
Adj R²  : 0.1618
RMSE    : 0.1812
MAE     : 0.1125
CV_R²   : -556.969

Decision tree
R²      : 0.8694
Adj R²  : 0.8378
RMSE    : 0.0797
MAE     : 0.0216
CV_R²   : -712.5693

Random forest
R²      : 0.9095
Adj R²  : 0.8876
RMSE    : 0.0663
MAE     : 0.0309
CV_R²   : -317.8028

Support vector machines
R²      : 0.7689
Adj R²  : 0.7132
RMSE    : 0.106
MAE     : 0.0735
CV_R²   : -496.836

Gradient boosting
R²      : 0.9066
Adj R²  : 0.8841
RMSE    : 0.0674
MAE     : 0.0226
CV_R²   : -301.3706


In [8]:
# ============================================================
# CREATE RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "R2",
        "Adjusted_R2",
        "RMSE",
        "MAE",
        "CV_R2"
    ]
)

# Ranking

results_df["Rank"] = (
        results_df["R2"].rank(
            ascending=False
        )
)

results_df = results_df.sort_values(
    "Rank"
)

results_df

,Model,R2,Adjusted_R2,RMSE,MAE,CV_R2,Rank
3,Random forest,0.909476,0.887626,0.066339,0.030891,-317.802759,1.0
5,Gradient boosting,0.906641,0.884106,0.067370,0.022586,-301.370631,2.0
2,Decision tree,0.869363,0.837830,0.079693,0.021649,-712.569275,3.0
4,Support vector machines,0.768948,0.713177,0.105984,0.073457,-496.836017,4.0
0,Multiple linear regression,0.393371,0.246944,0.171731,0.105903,-670.988026,5.0
1,LASSO regression,0.324804,0.161825,0.181176,0.112473,-556.969009,6.0


In [9]:
# ============================================================
# BEST MODEL
# ============================================================

best_model = results_df.iloc[0]

print("Best Model")

best_model

Best Model


Model          Random forest
R2                  0.909476
Adjusted_R2         0.887626
RMSE                0.066339
MAE                 0.030891
CV_R2            -317.802759
Rank                     1.0
Name: 3, dtype: object

In [10]:
# ============================================================
# EXPORT TO EXCEL
# ============================================================

os.makedirs("output/Sheet1", exist_ok=True)

with pd.ExcelWriter(
    "output/Sheet1/MIC90_Results.xlsx",
    engine="openpyxl"
) as writer:

    results_df.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    workbook = writer.book
    worksheet = writer.sheets["Results"]

    for cell in worksheet[1]:
        cell.font = Font(
            bold=True
        )
        
        cell.border = Border(
            left=Side(style="thin"),
            right=Side(style="thin"),
            top=Side(style="thin"),
            bottom=Side(style="thin")
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center"
        )

    for row in worksheet.iter_rows():

        for cell in row:

            cell.border = Border(
                left=Side(style="thin"),
                right=Side(style="thin"),
                top=Side(style="thin"),
                bottom=Side(style="thin")
            )

print("Results Excel Saved Successfully")

Results Excel Saved Successfully


In [11]:
# ============================================================
# EXPORT PREDICTIONS
# ============================================================

predictions_df.to_excel(
    "output/Sheet1/MIC90_Predictions.xlsx",
    index=False
)

print("Prediction File Saved")

Prediction File Saved
